In [ ]:
import numpy as np
import pandas as pd

# Generate data for NARX 1 & NARX 2

## NARX 1

In [ ]:
# NARX 1

def sim_narx1(n, sigmau, sigmaw, seed):
    y= np.zeros((n, 2))
    np.random.seed(seed)
    u = np.random.normal(0, sigmau, n)
    w = np.random.normal(0, sigmaw, (n, 2))

    y[1, :] = w[1, :] # k = 0

    for k in range(1, n - 1):
        y[k + 1, 0] = 0.5 * y[k - 1, 1] + np.sin(y[k, 1]) + 0.3 * u[k - 1] + w[k + 1, 0]

        y[k + 1, 1] = 0.5 * y[k - 1, 0] + np.sin(y[k, 0]) + 0.2 * u[k] + w[k + 1, 1]
    
    return u, y

#NARX 2
def sim_narx2(n, sigmau, sigmaw, seed):
    np.random.seed(seed)
    u = np.random.normal(0, sigmau, (n, 2))
    y = np.zeros((n, 2))
    w = np.random.normal(0, sigmaw, (n, 2))

    #y[0:3, :] = 0 #set initial conditions to 0 to avoid any instabily from the start
    y[0:3] = np.random.normal(0, 0.5, (3,2))
    
    for k in range(2, n - 1):
        top1 = (y[k, 0] * y[k - 1, 0] * y[k - 2, 0] * (y[k - 2, 0] - 1) * u[k - 1, 1]) + u[k, 1]
        bot1 = 1 + (y[k - 1, 1]**2) + (y[k - 2, 1]**2)
        y[k + 1, 0] = (top1 / bot1) + w[k + 1, 0]

        top2 = (y[k, 1] * y[k - 1, 1] * y[k - 2, 1] * (y[k - 2, 1] - 1) * u[k - 1, 0]) + u[k, 0]
        bot2 = 1 + (y[k - 1, 0]**2) + (y[k - 2, 0]**2)
        y[k + 1, 1] = (top2 / bot2) + w[k + 1, 1]
        
    return u, y

# Pre-processing

In [ ]:
u_narx1, y_narx1 = sim_narx1(1000, 1, 0.1, 42) #sigmau needs to be at least =1 bc if it's smaller y1 & y2 are too correlated
u_narx2, y_narx2 = sim_narx2(1000, 0.0001, 0.00001, 42) #sigmau & sigmaw need to be small, otherwise y explodes

In [ ]:
#turn generated data into dataframe so it's easier to use later on
import pandas as pd

def narx1_to_dataframe(u, y):
    narx1 = pd.DataFrame({
        "u": u_narx1,
        "y1": y_narx1[:, 0],
        "y2": y_narx1[:, 1]
    })
    return narx1

def narx2_to_dataframe(u, y):
    narx2 = pd.DataFrame({
        "u1": u_narx2[:, 0],
        "u2": u_narx2[:, 1],
        "y1": y_narx2[:, 0],
        "y2": y_narx2[:, 1],
    })
    return narx2

narx1 = narx1_to_dataframe(u_narx1, y_narx1)
narx2 = narx2_to_dataframe(u_narx2, y_narx2)

In [ ]:
import numpy as np

def monte_carlo_narx2(n_runs=50, n=1000, sigmau=.1, sigmaw=0.01):
    all_y = []

    for seed in range(n_runs):
        u, y = sim_narx2(n, sigmau, sigmaw, seed)
        all_y.append(y)

    all_y = np.concatenate(all_y, axis=0)
    return all_y

In [ ]:
y_mc = monte_carlo_narx2()

print("Min / Max :", np.min(y_mc), np.max(y_mc))
print("Percentiles :", np.percentile(y_mc, [1, 5, 50, 95, 99]))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(y_mc[:500])
plt.title("Trajectoire brute (Monte Carlo)")
plt.show()

In [ ]:
for n_runs in [10, 20, 50, 100]:
    y_mc = monte_carlo_narx2(n_runs=n_runs)
    print(n_runs, np.percentile(y_mc, [1, 99]))

In [ ]:
narx2

# Parameters selection approach

## NARX1

#### 1st approach: PACF
The Partial Autocorrelation Function (PACF) is used to guide the selection of the output lag parameter na in the NARX1 model. The main advantage of the PACF is that it measures the direct influence of past values of a signal on its current value, while removing the effect of intermediate lags.

In our case, this is useful because we want to identify which past outputs y(k-d) have a significant and independent contribution to the current output y(k). The PACF is that it won't be able to see the interaction between y1 & y2 but will give us idea of candidates to use for na in our grid search.

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import pacf
from statsmodels.graphics.tsaplots import plot_pacf


pacf_y1 = pacf(narx1["y1"], nlags=10)
pacf_y2 = pacf(narx1["y2"], nlags=10)

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PACF y1
plot_pacf(narx1["y1"], lags=15, ax=axes[0])
axes[0].set_title("PACF of y1")

# PACF y2
plot_pacf(narx1["y2"], lags=15, ax=axes[1])
axes[1].set_title("PACF of y2")

plt.tight_layout()
plt.show()


PACF of y1: ~lag 4 seems to be significant (2 is borderline, but still some influence)

PACF of y2: ~lag 4 seems to be significant (1-2 are small though)

Our candidates for na (output lag) -> [1:4] but PACF only takes individual effect of y(i) on itself -> we need to see the effect of y1 on y2 and vice versa

#### 2nd approach: CCF 

The Cross-Correlation Function (CCF) is used to guide the selection of the input-related parameters d and nb for the model. The CCF measures the correlation between the input and the output at different lags, which allows us to identify how the inputs influence the outputs over time.

Instead of using the CCF from statsmodels, we create our own CCF function because the statsmodels function calculates the corr(x(k−l),y(k)), but what we need is corr(x(k−l),y(k+1)). The function doesn't handle shifting the data.

In [ ]:
def compute_ccf(u, y, max_lag):
    ccf_values = []
    
    #we shift y by one so we look at y(k+1) and u(k)
    u = u[:-1]
    y = y[1:]
    
    for lag in range(max_lag + 1):
        if lag == 0:
            corr = np.corrcoef(u, y)[0,1]
        else:
            corr = np.corrcoef(u[:-lag], y[lag:])[0,1]
        
        ccf_values.append(corr)
    
    return np.array(ccf_values)

In [ ]:
u  = narx1["u"].values
y1 = narx1["y1"].values
y2 = narx1["y2"].values

ccf_u_y1 = compute_ccf(u, y1, max_lag=15)
ccf_u_y2 = compute_ccf(u, y2, max_lag=15)
ccf_y2_y1 = compute_ccf(y2, y1, max_lag=15)
ccf_y1_y2 = compute_ccf(y1, y2, max_lag=15)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# u → y1
axes[0, 0].stem(range(len(ccf_u_y1)), ccf_u_y1)
axes[0, 0].set_title("CCF: u -> y1")
axes[0, 0].set_xlabel("Lag")
axes[0, 0].set_ylabel("Correlation")

# u → y2
axes[0, 1].stem(range(len(ccf_u_y2)), ccf_u_y2)
axes[0, 1].set_title("CCF: u -> y2")
axes[0, 1].set_xlabel("Lag")
axes[0, 1].set_ylabel("Correlation")

# y2 → y1
axes[1, 0].stem(range(len(ccf_y2_y1)), ccf_y2_y1)
axes[1, 0].set_title("CCF: y2 -> y1")
axes[1, 0].set_xlabel("Lag")
axes[1, 0].set_ylabel("Correlation")

# y1 → y2
axes[1, 1].stem(range(len(ccf_y1_y2)), ccf_y1_y2)
axes[1, 1].set_title("CCF: y1 -> y2")
axes[1, 1].set_xlabel("Lag")
axes[1, 1].set_ylabel("Correlation")

plt.tight_layout()
plt.show()



The first lag where correlation becomes significative will help us choose the delay

u -> y1 : the first significant effect is seen at lag 1 (with secondary effect at lag 2) --> nb_y1 = [1, 2]
u -> y2 : the first significant effect is seen at lag 0 (dominant) --> nb_y2 = [0, 1, 2, 3]


y2 -> y1 : there are significant effects at lags 0 and 1 --> na_y1 = [0, 1]
y1 -> y2 : significant effects at lag 1 --> na_y2 ~ = [1, 2, 3]

--> delay = d ~ [0,1], na ~ = [1, 2, 3, 4] (we take 4 to not be too strict)

#### Parameters estimation

We put all of our candidates parameters in variables to use in the grid search.

In [ ]:
#######    NARX 1 - PARAMETERS ESTIMATION
# OUTPUT LAGS
na_y1_grid_narx1 = [1,2,3,4]
na_y2_grid_narx1 = [1,2,3,4]

# INPUT LAGS
nb_y1_grid_narx1 = [1,2]
nb_y2_grid_narx1 = [0,1,2,3]

# DELAYS
d_y1_grid_narx1  = [0,1]
d_y2_grid_narx1  = [0,1]

# NARX2

#### STEP 1: create lagged dataset + targets and features

For the NARX2 model, a different approach was required since the model is nonlinear. Tools such as PACF and CCF are mainly suited for linear relationships and therefore are not appropriate to capture nonlinear dependencies between inputs and outputs.

We therefore adopted a data-driven feature engineering approach. First, we constructed a dataset including multiple lags of the inputs and outputs. 

In [ ]:
import pandas as pd

def lagged_dataset(u1, u2, y1, y2, max_lag=3):
    
    data = pd.DataFrame({
    "u1": u1,
    "u2": u2,
    "y1": y1,
    "y2": y2
    })

    lagged_data = pd.DataFrame()
    
    for var in ["u1", "u2", "y1", "y2"]:
        for lag in range(1, max_lag + 1):
            lagged_data[f"{var}_lag{lag}"] = data[var].shift(lag)

    # Targets
    lagged_data["y1_target"] = data["y1"].shift(-1)
    lagged_data["y2_target"] = data["y2"].shift(-1)

    # Drop NaNs
    lagged_data = lagged_data.dropna().reset_index(drop=True)

    # Séparation
    X = lagged_data.drop(columns=["y1_target", "y2_target"])
    y1_target = lagged_data["y1_target"]
    y2_target = lagged_data["y2_target"]

    return X, y1_target, y2_target

In [ ]:
u1_narx2 = narx2["u1"]
u2_narx2 = narx2["u2"]
y1_narx2 = narx2["y1"]
y2_narx2 = narx2["y2"]

X, y1_target, y2_target = lagged_dataset(u1_narx2, u2_narx2, y1_narx2, y2_narx2, max_lag=3)

#### STEP 2: polynomial features 

We generate polynomial features in order to capture nonlinear interactions between variables, including higher-order terms (e.g., quadratic and cubic interactions).

However, this step significantly increases the number of features, which can negatively impact the model by introducing noise and overfitting. It was therefore necessary to perform feature selection.

To do so, we combined two methods: ranking with mutual information, and wrapper with forward selection.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=3, include_bias=False)

X_poly = poly.fit_transform(X)

feature_names = poly.get_feature_names_out(X.columns)

print("Original shape:", X.shape, ", nb features:", X.shape[1])
print("Polynomial shape:", X_poly.shape, ", nb features:", X_poly.shape[1])

#### STEP 3: mutual information on the features 

We use the filter method, mutual information on our polynomial features. This is used to rank the features based on their dependency with the target. It allows us to keep the most informative variables, including nonlinear relationships.

In [ ]:
from sklearn.feature_selection import mutual_info_regression

mi_y1 = mutual_info_regression(X_poly, y1_target)
mi_y2 = mutual_info_regression(X_poly, y2_target)

In [ ]:
import numpy as np

indexes_y1 = np.argsort(mi_y1)[::-1]
mi_y1_sorted = mi_y1[indexes_y1]

indexes_y2 = np.argsort(mi_y2)[::-1]
mi_y2_sorted = mi_y2[indexes_y2]

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Graphique y1
axs[0].plot(mi_y1_sorted)
axs[0].set_title("Mutual Information ranking (y1)")
axs[0].set_xlabel("Feature rank")
axs[0].set_ylabel("MI")

# Graphique y2
axs[1].plot(mi_y2_sorted)
axs[1].set_title("Mutual Information ranking (y2)")
axs[1].set_xlabel("Feature rank")
axs[1].set_ylabel("MI")

plt.tight_layout()
plt.show()



We can see that a lot of features are ~0, but it's hard to tell exactly how many to take by looking at the graph, so we'll use cumulative MI. 
--> We'll keep the features that gives us 90% of the information

In [ ]:
def find_k_cumulative(mi_sorted, ratio=0.9):
    cum = np.cumsum(mi_sorted)
    cum /= cum[-1]
    return np.where(cum >= ratio)[0][0]

k_y1 = find_k_cumulative(mi_y1_sorted)
k_y2 = find_k_cumulative(mi_y2_sorted)

print("K_y1 (cumulative):", k_y1)
print("K_y2 (cumulative):", k_y2)

#save the features' indexes and name for later use
selected_idx_y1 = indexes_y1[:88]
selected_idx_y2 = indexes_y2[:73]

feature_names = poly.get_feature_names_out()

selected_names_y1 = feature_names[selected_idx_y1]
selected_names_y2 = feature_names[selected_idx_y2]

#### STEP 4: wrapper method on selected features 
We use a wrapper method (forward selection). We starting from an empty model, and we add features one by one. At each step, we evaluate the model performance using the Normalized Mean Squared Error (NMSE). This allows us to observe how the error evolves as features are added and to identify the optimal number of variables that minimizes the prediction error.

In [ ]:
#features selected by MI
X_y1 = X_poly[:, selected_idx_y1]
X_y2 = X_poly[:, selected_idx_y2]

# Targets
y1 = y1_target
y2 = y2_target


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm

X = X_y1          # features after MI
Y = y1            # target
n = X.shape[1]
max_features = 15  # max features to select

# NMSE
def nmse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2) / np.var(y_true)

# CROSS VALIDATION (TIME SERIES)
kf = TimeSeriesSplit(n_splits=5)


# FORWARD SELECTION
selected = []
CV_errs = []
std_errs = []

for round_i in tqdm(range(max_features)):
    candidates = list(set(range(n)) - set(selected))
    
    CV_err_temp = []
    std_temp = []
    
    for c in candidates:
        features_to_include = selected + [c]
        fold_errors = []
        
        for train_index, test_index in kf.split(X):
            X_tr = X[train_index][:, features_to_include]
            X_ts = X[test_index][:, features_to_include]
            
            Y_tr = Y[train_index]
            Y_ts = Y[test_index]
            
            model = LinearRegression()
            model.fit(X_tr, Y_tr)
            
            Y_hat_ts = model.predict(X_ts)
            fold_errors.append(nmse(Y_ts, Y_hat_ts))
        
        CV_err_temp.append(np.mean(fold_errors))
        std_temp.append(np.std(fold_errors))
    
    best_idx = np.argmin(CV_err_temp)
    best_candidate = candidates[best_idx]
    
    selected.append(best_candidate)
    
    CV_errs.append(CV_err_temp[best_idx])
    std_errs.append(std_temp[best_idx])
    
    print(f"Step {round_i+1}: added feature {best_candidate}, NMSE={CV_err_temp[best_idx]:.6f}")


# FINAL FEATURES 
selected_features_names = selected_names_y1[selected]

print("\nSelected features (y1):")
print(selected_features_names)


# PLOT
x = list(range(1, len(CV_errs)+1))
errors = np.array(CV_errs)
stds = np.array(std_errs)

plt.figure(figsize=(10, 6))
plt.plot(x, errors, label="Average NMSE")
plt.fill_between(x, errors - stds, errors + stds, alpha=0.2, label="Std NMSE")
plt.xlabel("Number of selected features")
plt.ylabel("NMSE")
plt.title("Forward Selection + CV (y1)")
plt.legend()
plt.grid(True)
plt.show()

The number of selected features was determined using a forward selection procedure combined with time-series cross-validation. At each iteration, one candidate feature is added to the current set of selected features. For each candidate, the model is trained and evaluated using a TimeSeriesSplit, which preserves the temporal structure of the data.

The performance is assessed using the Normalized Mean Squared Error (NMSE). At each step, the feature that minimizes the average NMSE across the validation folds is selected. In addition, the standard deviation of the NMSE is computed to evaluate the stability of the model.

From the resulting curve (NMSE vs number of features), we observe that:
* the NMSE slightly decreases in the first iterations,
* then reaches a minimum around 4–6 features,
* and finally starts to increase again while the variance also grows.

Beyond this point, adding more features does not significantly improve the performance and even leads to a slight degradation of the NMSE. Therefore, the optimal number of features (k) is chosen in this elbow region, where the model achieves a good trade-off between accuracy and robustness.

Therefore, k is chosen around this region (≈ 5 features).

##### **Selected features interpretation**

The selected features show that the system depends on both short-term (lag 1) and delayed dynamics (lag 3) of the output y1.

The inputs also play an important role:

u2 has a strong influence across several lags (1 to 3), including nonlinear effects (e.g., ( u2_lag3^2 )),
u1 mainly affects the system with a delayed impact (lags 2 and 3).

Interaction terms are present, showing that the system is nonlinear, where the effect of inputs depends on past outputs.

In [ ]:
final_k = 5
final_idx = selected[:final_k]
final_features = selected_names_y1[final_idx]

print(final_features)

In [ ]:
X = X_y2
Y = y2
selected_names_y = selected_names_y2

#### y2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm


X = X_y2          # features after MI
Y = y2            # target
n = X.shape[1]
max_features = 15  # max features to select

# NMSE
def nmse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2) / np.var(y_true)

# CROSS VALIDATION (TIME SERIES)
kf = TimeSeriesSplit(n_splits=5)


# FORWARD SELECTION

selected = []
CV_errs = []
std_errs = []

for round_i in tqdm(range(max_features)):
    candidates = list(set(range(n)) - set(selected))
    
    CV_err_temp = []
    std_temp = []
    
    for c in candidates:
        features_to_include = selected + [c]
        fold_errors = []
        
        for train_index, test_index in kf.split(X):
            X_tr = X[train_index][:, features_to_include]
            X_ts = X[test_index][:, features_to_include]
            
            Y_tr = Y[train_index]
            Y_ts = Y[test_index]
            
            model = LinearRegression()
            model.fit(X_tr, Y_tr)
            
            Y_hat_ts = model.predict(X_ts)
            fold_errors.append(nmse(Y_ts, Y_hat_ts))
        
        CV_err_temp.append(np.mean(fold_errors))
        std_temp.append(np.std(fold_errors))
    
    best_idx = np.argmin(CV_err_temp)
    best_candidate = candidates[best_idx]
    
    selected.append(best_candidate)
    
    CV_errs.append(CV_err_temp[best_idx])
    std_errs.append(std_temp[best_idx])
    
    print(f"Step {round_i+1}: added feature {best_candidate}, NMSE={CV_err_temp[best_idx]:.6f}")


# FINAL FEATURES 
selected_features_names = selected_names_y1[selected]

print("\nSelected features (y2):")
print(selected_features_names)


# PLOT
x = list(range(1, len(CV_errs)+1))
errors = np.array(CV_errs)
stds = np.array(std_errs)

plt.figure(figsize=(10, 6))
plt.plot(x, errors, label="Average NMSE")
plt.fill_between(x, errors - stds, errors + stds, alpha=0.2, label="Std NMSE")
plt.xlabel("Number of selected features")
plt.ylabel("NMSE")
plt.title("Forward Selection + CV (y2)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
final_k = 5
final_idx_y2 = selected[:final_k]
final_features_y2 = selected_names_y2[final_idx_y2]

print(final_features_y2)

From the NMSE curve, the error decreases slightly in the first iterations and reaches a minimum around 4–6 features, after which it starts to increase again. At the same time, the variability (std) grows as more features are added.

This indicates that adding more features beyond this point does not improve performance and may lead to overfitting. Therefore, k was chosen in this region (≈ 5 features).


The selected features show that y2 is mainly driven by short-term dynamics (lag 1), with some contribution from lag 2 and lag 3.

Both inputs play a role:
* u2 influences the system across several lags (1 to 3)
* u1 contributes both linearly and nonlinearly

The presence of interaction terms shows that the system is nonlinear and coupled, where inputs and past outputs interact.

#### Parameters estimation

In [ ]:
#######    NARX 2 - PARAMETERS ESTIMATION

# OUTPUT LAGS
na_y1_grid_narx2 = [1,2,3]
na_y2_grid_narx2 = [1,2,3]

# INPUT LAGS
nb_y1_grid_narx2 = [1,2,3]
nb_y2_grid_narx2 = [1,2,3]

# DELAYS
d_y1_grid_narx2 = [1, 2]
d_y2_grid_narx2 = [1, 2]

### MIMO and multi-step-ahead strategy selection approach

#### A) MIMO nature of the task
We aim to predict two outputs simultaneously. To simplify the modelling process, we chose to decompose the problem into two separate models, meaning each output is modeled independently using the same set of inputs and past outputs (y1, y2, u1 & u2). 

This approach allows us to better understand the individual dynamics of each output and to tune the model structure more effectively, while still capturing the interactions through the inclusion of cross-output lags.

#### B) multi-step-ahead strategy

Regarding the multi-step-ahead prediction, the main challenge comes from the long prediction horizon. In such settings, predictions are recursively fed back into the model, which can lead to error accumulation and, eventually, divergence. We expect the first predictions to be relatively accurate, while the quality may degrade as we move further into the future.

To address this, we first evaluate the selected model over multiple prediction horizons and assess its performance using metrics such as the NMSE. This allows us to observe how the prediction error evolves over time and to determine whether the model remains reliable for long-term forecasting or not. Based on this, we can also identify up to which horizon the model can be considered "trustworthy".

To improve performance at longer horizons, we consider regularization techniques such as Ridge regression, which help reduce overfitting and stabilize the predictions. 

An alternative approach is to train different models for different prediction horizons, allowing each model to specialize and potentially improve performance over longer horizons.